In [2]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0], 
                                  [0.0, 0.9, 0.1, 0.0, 0.0], 
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]]) 

emission_nuc_codes = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00], 
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]]) 

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"

In [3]:
def get_log_prob_for_state_path(state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

# CTTCATGTGAAAGCAGACGTAAGTCA 
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print(f"Path k1: {k1}")

k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print(f"Path k2: {k2}")

k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print(f"Path k3: {k3}")

k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print(f"Path k4: {k4}")

k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print(f"Path k5: {k5}")

k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print(f"Path k6: {k6}")

only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print(f"Path only_E: {only_E}")

Path k1: -43.89740030179307
Path k2: -43.45111319916465
Path k3: -43.944833355027704
Path k4: -42.58225552052512
Path k5: -41.21967768602254
Path k6: -41.713397841885595
Path only_E: -40.98025137355685


In [4]:
# Initiate two matrices: 
# viterbi_value_matrix: to store the log probabilities 
# viterbi_trace_matrix: to store the path

seq_len = len(query_sequence)
num_states = len(states)

# Initialize with negative infinity to represent log(0)
viterbi_value_matrix = np.full((num_states, seq_len), -np.inf)
viterbi_trace_matrix = np.zeros((num_states, seq_len), dtype=int)

# Initialize the first column (assuming we start in Exon 'E' based on your evaluation functions)
first_nuc_idx = emission_nuc_codes[query_sequence[0]]
# Log prob of emitting first nucleotide in state E (index 1)
viterbi_value_matrix[1, 0] = math.log(emission_probs[1][first_nuc_idx])

# Initialize the first trace column with their own indices as requested
for i in range(num_states):
    viterbi_trace_matrix[i, 0] = i

In [5]:
def calculate_prob_for_a_node(cur_state, col_idx, sequence):
    max_val = -np.inf
    best_prev_state = 0
    
    nuc = sequence[col_idx]
    nuc_idx = emission_nuc_codes[nuc]
    
    # Calculate the emission probability
    emit_prob = emission_probs[cur_state][nuc_idx]
    
    if emit_prob == 0:
        return -np.inf, 0 # Impossible to emit
        
    log_emit_prob = math.log(emit_prob)
    
    # Loop over all possible previous states
    for prev_state in range(num_states):
        trans_prob = state_transition_prob[prev_state][cur_state]
        
        if trans_prob == 0:
            continue # Impossible transition
            
        prev_val = viterbi_value_matrix[prev_state, col_idx - 1]
        
        if prev_val == -np.inf:
            continue # Previous state was unreachable
            
        log_trans_prob = math.log(trans_prob)
        
        # Calculate current probability path
        current_prob = prev_val + log_trans_prob + log_emit_prob
        
        # Update max value and traceback index
        if current_prob > max_val:
            max_val = current_prob
            best_prev_state = prev_state
            
    return max_val, best_prev_state

In [6]:
# Write for loops to iterate over the whole Viterbi Value matrix. 
# Each time, call the function
for col_idx in range(1, seq_len):
    for cur_state in range(num_states):
        max_val, best_prev_state = calculate_prob_for_a_node(cur_state, col_idx, query_sequence)
        
        viterbi_value_matrix[cur_state, col_idx] = max_val
        viterbi_trace_matrix[cur_state, col_idx] = best_prev_state

In [7]:
# Write a function to trace the state path that gave the maximum probability. 

# 1. Find the max value in the last column
last_col_idx = seq_len - 1
best_last_state = np.argmax(viterbi_value_matrix[:, last_col_idx])
max_prob = viterbi_value_matrix[best_last_state, last_col_idx]

print(f"Maximum Probability Found: {max_prob}")

# 2. Traceback sequence
optimal_path_indices = [best_last_state]
current_state = best_last_state

for col_idx in range(last_col_idx, 0, -1):
    prev_state = viterbi_trace_matrix[current_state, col_idx]
    optimal_path_indices.append(prev_state)
    current_state = prev_state

# 3. Reverse the array to get the sequence from left to right
optimal_path_indices.reverse()

# 4. Map numerical states back to letters
optimal_state_path = "".join([id2state[idx] for idx in optimal_path_indices])

print(f"Query Sequence : {query_sequence}")
print(f"Most Likely Path: {optimal_state_path}")

Maximum Probability Found: -38.677666280562796
Query Sequence : CTTCATGTGAAAGCAGACGTAAGTCA
Most Likely Path: EEEEEEEEEEEEEEEEEEEEEEEEEE
